https://github.com/inceptezwd37we48/iz_we48_databricks_repo/blob/main/lakeflow_learning/lakeflow_ingestion_cloudfile_autoloader_1.ipynb

In [0]:
# variable definition with paths
cloudsrc="/Volumes/lakehousecat1/deltadb/deltalake/lakeflow/landing"
bronzetgt ="/Volumes/lakehousecat1/deltadb/deltalake/lakeflow/bronze"
chkpointlocation ="/Volumes/lakehousecat1/deltadb/deltalake/lakeflow/_checkpoint"
#chkpointlocation ="/Volumes/lakehousecat1/deltadb/deltalake/lakeflow1/_checkpoint1"
schemalocation="/Volumes/lakehousecat1/deltadb/deltalake/lakeflow/_schema"

# if folders and sub folders already exist to remove those
dbutils.fs.rm(cloudsrc,True)
dbutils.fs.rm(bronzetgt,True)
dbutils.fs.rm(chkpointlocation,True)
dbutils.fs.rm(schemalocation,True)

# create folders
dbutils.fs.mkdirs(cloudsrc)
dbutils.fs.mkdirs(bronzetgt)
dbutils.fs.mkdirs(chkpointlocation)
dbutils.fs.mkdirs(schemalocation)

#read stream 

df=spark.readStream.format("cloudFiles")\
   .option("cloudFiles.format","csv") \
   .option("cloudFiles.MaxFilesPerTrigger","1")\
   .option("inferColumnTypes","True")\
   .option("cloudFiles.schemaEvolutionMode","addNewColumns")    \
   .option("checkpointLocation",chkpointlocation)\
   .option("cloudFiles.schemaLocation",schemalocation)\
   .option("Header",True)\
   .load(cloudsrc)


# spark.read.option(k,v).format("csv").load() - spark sql , batch mode 
# spark.readStream.option(k,v).format("csv").load()  - streaming mode , structure streaming -micro
# spark.readStream.format("cloudFiles")  - streaming mode , auto loader - microbatch
# streaming source , auto loader is provoiding 
# spark.read.format("parquet")  - batch mode / spark sql  
# spark.readStream.format("parquet") - streaming mode 
# cloudFiles - streaming file source , supports csv, json, avro,etc ..
# cloudFiles.format - specify file type (csv , json , parquet, avro , xml ...)
# cloudFiles.maxFilesPerTrigger - per fetch how many files it will bring from source 
# cloudFiles.inferColumnTypes -True( inferschema ), False (default all columns are string type )
# https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/schema
# cloudFiles.schemaLocation  - loc , schemma will be stored in this location , track the schema changes 
# checkpointLocation - checkpoint location for the stream , maintains the state of the stream 
# cloudFiles.schemaEvolutionMode - addNewColumns(default) , fail , rescue (coulnnameforcoruptedrecord=_rescued_data), none
# permissive + coulnnameforcoruptedrecord , dropmalfoormed , failfast 


In [0]:
# file creation

with open(cloudsrc+"/emp2.csv","w") as f:
    f.write("id,name,age\n")
    f.write("1,John,30\n")
    f.write("2,Jane,25\n")
    f.write("3,Mark,40\n")
    f.close

In [0]:

# trigger and save as table

df.writeStream.trigger(availableNow=True).\
    option("mergeSchema","True").\
    option("checkpointLocation",chkpointlocation).\
    option("cloudFiles.schemaLocation",schemalocation).\
    table("lakehousecat1.deltadb.emp_autoloader")

# trigger and save as file
df.writeStream.trigger(availableNow=True).\
    option("mergeSchema","True").\
    option("checkpointLocation",chkpointlocation).\
    option("cloudFiles.schemaLocation",schemalocation).\
    start(bronzetgt)

In [0]:
%sql


select * from lakehousecat1.deltadb.emp_autoloader

